# E2.1 — Extended Production (single run)

Extended-campaign production for the post-checkpoint sampling design. Same MD protocol as
the 100 ns diagnostic runs (E1.2) — Langevin 300 K / 1 ps^-1 / 2 fs, Monte Carlo barostat
1 bar, PME 1.0 nm, HBonds constraints, 1 ps save, OpenCL default precision — carried forward
unchanged so the extended trajectories stay directly comparable to the diagnostic set.

What is new here is length and start sourcing:
- **Length:** 200 ns (`NS`), the campaign baseline from the tau_int verdict.
- **Start:** an *extended* start (frame 0 of the system's `_prod.dcd`, the validated
  coordinate source) when `START_STRUCTURE = None`, or a *diverse* start when given a path to
  an extracted frame. Tonight's run is the extended-start replicate; the compact/diverse
  starts follow once the start-frame extractor is built.
- **Labelling:** runs go to `trajectories_extended/{system}_{NS}ns_{start}_r{rep}/`, separate
  from the 100 ns diagnostic runs in `trajectories/`, with a deterministic per-run seed.

A start-PE guard (|PE| < 1e7 kJ/mol) aborts on bad coordinates before the night is committed,
and the run self-verifies (frame count + finite last frame) and writes a metadata record with
a `verified` flag that E1.3 reads.

## 1 — Imports
**Out:** namespace.

In [1]:
import sys, time, json, hashlib
from pathlib import Path

import numpy as np
import mdtraj as md
from openmm import LangevinIntegrator, MonteCarloBarostat, Platform
from openmm.app import (AmberPrmtopFile, Simulation, DCDReporter,
                        StateDataReporter, PDBFile, PME, HBonds)
from openmm.unit import (kelvin, picosecond, picoseconds, bar,
                         nanometer, kilojoules_per_mole)

import openmm
print(f"OpenMM {openmm.__version__} | MDTraj {md.__version__}")
print("Platforms:", [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())])

OpenMM 8.5.1 | MDTraj 1.11.1
Platforms: ['Reference', 'CPU', 'OpenCL']


## 2 — Protocol constants (carried forward — do not edit)
Identical to the E1.2 production protocol. Changing anything here breaks comparability with the
diagnostic trajectories.

In [2]:
TEMPERATURE     = 300 * kelvin
FRICTION        = 1 / picosecond
TIMESTEP        = 0.002 * picoseconds
PRESSURE        = 1 * bar
CUTOFF          = 1.0 * nanometer
SAVE_INTERVAL   = 500     # steps -> 1 ps per frame
REPORT_INTERVAL = 5000    # steps -> 10 ps per log line
STDOUT_INTERVAL = 50000   # steps -> progress to notebook
PE_GUARD        = 1e7     # |start PE| above this aborts (kJ/mol)

## 3 — Run configuration (edit per run)
`START_STRUCTURE = None` uses the extended start (frame 0 of `_prod.dcd`). For diverse/compact
starts later, set it to a path to the extracted frame and set `START` to a matching label.

In [3]:
SYSTEM          = "GGE_glyceline"
NS              = 200
START           = "mid"
REP             = 1
PLATFORM        = "OpenCL"
START_STRUCTURE = "/Users/rossgibson/des-peptide-study/extension/start_frames/GGE_glyceline/GGE_glyceline_mid.pdb"

## 4 — Deterministic seed
Reproducible per (system, length, start, replicate).
**Out:** `SEED`.

In [4]:
key = f"{SYSTEM}|{NS}|{START}|{REP}"
SEED = int(hashlib.sha256(key.encode()).hexdigest(), 16) % (2**31)
print(f"run key: {key}\nseed: {SEED}")

run key: GGE_glyceline|200|mid|1
seed: 1769059031


## 5 — Paths
**Out:** resolved topology, start source, output paths.

In [5]:
PROJECT_DIR = Path("~/des-peptide-study").expanduser()
SYSTEMS_DIR = PROJECT_DIR / "systems"

def resolve(name):
    for cand in (SYSTEMS_DIR / SYSTEM / name, SYSTEMS_DIR / name):
        if cand.exists():
            return cand
    raise FileNotFoundError(f"{name} not found for {SYSTEM}")

PRMTOP   = resolve(f"{SYSTEM}.prmtop")
SRC_DCD  = resolve(f"{SYSTEM}_prod.dcd") if START_STRUCTURE is None else None

TAG      = f"{SYSTEM}_{NS}ns_{START}_r{REP}"
OUT_DIR  = PROJECT_DIR / "extension" / "trajectories_extended" / TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
DCD_OUT  = OUT_DIR / f"{TAG}.dcd"
LOG_OUT  = OUT_DIR / f"{TAG}_production.log"
PDB_OUT  = OUT_DIR / f"{TAG}_final.pdb"
META_OUT = OUT_DIR / f"{TAG}_metadata.json"

print(f"topology : {PRMTOP}")
print(f"start    : {'frame 0 of '+SRC_DCD.name if START_STRUCTURE is None else START_STRUCTURE}")
print(f"output   : {OUT_DIR}")
if DCD_OUT.exists():
    print("WARNING: a DCD already exists at this tag and would be overwritten.")

topology : /Users/rossgibson/des-peptide-study/systems/GGE_glyceline/GGE_glyceline.prmtop
start    : /Users/rossgibson/des-peptide-study/extension/start_frames/GGE_glyceline/GGE_glyceline_mid.pdb
output   : /Users/rossgibson/des-peptide-study/extension/trajectories_extended/GGE_glyceline_200ns_mid_r1


## 6 — Load starting coordinates
Extended start reads frame 0 of `_prod.dcd` (enforces prmtop atom order); a diverse start reads
the provided structure. **Out:** `start_xyz`, `start_box`.

In [6]:
if START_STRUCTURE is None:
    frame = md.load_frame(str(SRC_DCD), 0, top=str(PRMTOP))
else:
    frame = md.load(str(START_STRUCTURE), top=str(PRMTOP))[0]

start_xyz = frame.xyz[0]                 # nm, (N,3)
start_box = frame.unitcell_vectors[0]    # nm, (3,3)
print(f"atoms: {frame.n_atoms} | box diag (nm): {np.round(np.diag(start_box),3)}")

atoms: 2894 | box diag (nm): [3.059 3.059 3.059]


## 7 — Build system and context
Creates the system from the topology, sets coordinates and box, and checks the start PE before
anything runs. **Out:** `sim` ready to run; aborts if PE is unphysical.

In [7]:
prmtop = AmberPrmtopFile(str(PRMTOP))
system = prmtop.createSystem(nonbondedMethod=PME, nonbondedCutoff=CUTOFF, constraints=HBonds)
system.addForce(MonteCarloBarostat(PRESSURE, TEMPERATURE))

integrator = LangevinIntegrator(TEMPERATURE, FRICTION, TIMESTEP)
integrator.setRandomNumberSeed(SEED)

platform = Platform.getPlatformByName(PLATFORM)
sim = Simulation(prmtop.topology, system, integrator, platform)
sim.context.setPositions(start_xyz * nanometer)
sim.context.setPeriodicBoxVectors(start_box[0]*nanometer, start_box[1]*nanometer, start_box[2]*nanometer)

pe = sim.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
print(f"start PE: {pe:,.1f} kJ/mol")
assert abs(pe) < PE_GUARD, f"start PE {pe:.3e} kJ/mol exceeds guard -- suspect coordinates, aborting"
sim.context.setVelocitiesToTemperature(TEMPERATURE, SEED)
print("PE guard passed; velocities assigned.")

start PE: -31,307.2 kJ/mol
PE guard passed; velocities assigned.


## 8 — Run
Attaches reporters (1 ps DCD, 10 ps log, periodic notebook progress) and runs `NS` ns.
**In:** `sim`. **Out:** trajectory + log on disk; `wall_h`, `ns_per_day`.

In [8]:
N_STEPS  = int(NS * 500_000)               # NS ns * 500k steps/ns at 2 fs
EXPECTED = N_STEPS // SAVE_INTERVAL         # frames

sim.reporters.append(DCDReporter(str(DCD_OUT), SAVE_INTERVAL))
sim.reporters.append(StateDataReporter(str(LOG_OUT), REPORT_INTERVAL, step=True, time=True,
                                       potentialEnergy=True, temperature=True, density=True, speed=True))
sim.reporters.append(StateDataReporter(sys.stdout, STDOUT_INTERVAL, step=True, time=True,
                                       speed=True, remainingTime=True, totalSteps=N_STEPS))

print(f"running {NS} ns ({N_STEPS:,} steps) -> {EXPECTED:,} frames\n")
t0 = time.time()
sim.step(N_STEPS)
wall_h = (time.time() - t0) / 3600
ns_per_day = NS * 24 / wall_h
print(f"\ndone: {wall_h:.2f} h | {ns_per_day:.1f} ns/day")

running 200 ns (100,000,000 steps) -> 200,000 frames

#"Step","Time (ps)","Speed (ns/day)","Time Remaining"
50000,99.99999999994834,0,--
100000,200.00000000022686,696,6:53:20
150000,300.00000000070435,695,6:53:40
200000,400.00000000118183,695,6:53:44
250000,500.0000000016593,695,6:53:29
300000,599.9999999996356,695,6:53:14
350000,699.999999997271,694,6:53:40
400000,799.9999999949063,693,6:53:47
450000,899.9999999925416,693,6:53:52
500000,999.9999999901769,692,6:53:52
550000,1099.9999999878123,692,6:53:48
600000,1199.9999999854476,692,6:53:42
650000,1299.999999983083,692,6:53:41
700000,1399.9999999807183,691,6:53:38
750000,1499.9999999783536,691,6:53:35
800000,1599.9999999759889,691,6:53:29
850000,1699.9999999736242,691,6:53:23
900000,1799.9999999712595,690,6:53:21
950000,1899.9999999688948,690,6:53:16
1000000,1999.9999999665301,690,6:53:11
1050000,2099.9999999641655,690,6:53:06
1100000,2199.9999999618008,690,6:53:02
1150000,2299.999999959436,689,6:52:56
1200000,2399.9999999570714,689,6

## 9 — Save, verify, record
Writes the final coordinates, checks frame count and a finite last frame, and saves metadata
with a `verified` flag for E1.3 to read. **Out:** `*_final.pdb`, `*_metadata.json`.

In [9]:
with open(PDB_OUT, "w") as fh:
    PDBFile.writeFile(prmtop.topology, sim.context.getState(getPositions=True).getPositions(), fh)

with md.formats.DCDTrajectoryFile(str(DCD_OUT)) as f:
    n_frames = len(f)
last = md.load_frame(str(DCD_OUT), n_frames - 1, top=str(PRMTOP))
finite = bool(np.isfinite(last.xyz).all())
verified = (n_frames == EXPECTED) and finite

meta = {
    "system": SYSTEM, "ns": NS, "start": START, "replicate": REP, "tag": TAG,
    "seed": SEED, "platform": PLATFORM, "n_atoms": int(frame.n_atoms),
    "n_steps": N_STEPS, "expected_frames": EXPECTED, "n_frames": int(n_frames),
    "last_frame_finite": finite, "verified": bool(verified),
    "start_pe_kJ_mol": round(float(pe), 1),
    "wall_h": round(wall_h, 2), "ns_per_day": round(ns_per_day, 1),
    "save_interval_steps": SAVE_INTERVAL, "start_structure": str(START_STRUCTURE),
}
META_OUT.write_text(json.dumps(meta, indent=2))

print(f"frames {n_frames}/{EXPECTED} | last finite {finite} | verified {verified}")
print("metadata:", META_OUT)

frames 200000/200000 | last finite True | verified True
metadata: /Users/rossgibson/des-peptide-study/extension/trajectories_extended/GGE_glyceline_200ns_mid_r1/GGE_glyceline_200ns_mid_r1_metadata.json
